# 02 — Predicting outage severity (Objective 1)

**Target:** peak % of customers out per (storm episode, county), as `log(1 + percent)` for regression and low (<1%) / medium (1–10%) / high (≥10%) for classification.
**Split:** train 2014–2019, validate 2020 (tuning / early stopping), test 2021–2022.
**Models:** storm-type baseline → linear / logistic → random forest → LightGBM. **SHAP** explains the LightGBM regressor.

In [ ]:
# Works on Colab (Drive) or locally (repo checkout)
import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    if os.environ.get("LIGHTSOUT_PROJECT"):  # a repo checkout, e.g. run with nbconvert inside Colab
        raise ImportError
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = pathlib.Path("/content/drive/MyDrive/LightsOut")
    ZIP = pathlib.Path("/content/drive/MyDrive/LightsOut_code.zip")
    if ZIP.exists():  # refresh code (src/, notebooks/) from the latest uploaded zip; data/ is untouched
        import zipfile; zipfile.ZipFile(ZIP).extractall(PROJECT)
    !pip -q install polars lifelines scikit-survival shap geopandas lightgbm
except ImportError:
    PROJECT = pathlib.Path(os.environ["LIGHTSOUT_PROJECT"]) if os.environ.get("LIGHTSOUT_PROJECT") else \
        pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightsout import config as C, viz
viz.setup()
df = pd.read_parquet(C.PROCESSED / "model_table.parquet")
print(f"{len(df):,} (storm episode, county) units, {df.fips.nunique():,} counties, years {df.year.min()}-{df.year.max()}")

In [ ]:
from lightsout import models_severity as S
print(df.groupby("split")["severity"].value_counts(normalize=True).unstack().round(3))
fit = S.fit_all(df)

In [ ]:
res_val, res_test = S.results_table(fit, "val"), S.results_table(fit, "test")
res_test.to_csv(C.REPORTS / "severity_results_test.csv")
(C.REPORTS / "severity_results_test.md").write_text(res_test.to_markdown())
res_val.to_csv(C.REPORTS / "severity_results_val.csv")
print("LightGBM trees (stage 1 early-stopped / final refit):", fit["n_iter"])
print("Decision weights for F1-tuned cutoffs (fit on 2020):", fit["decision_weights"])
print("Validation (2020, models trained 2014-2019)"); display(res_val)
print("Test (2021-2022, models refit on 2014-2020)"); display(res_test)

### Uncertainty, per-class performance and hyperparameters
Cluster bootstrap over storm episodes (500 resamples) gives 95% intervals for the test metrics.

In [ ]:
ci = S.bootstrap_ci(fit, "test", n_boot=500)
ci.to_csv(C.REPORTS / "severity_ci_test.csv"); (C.REPORTS / "severity_ci_test.md").write_text(ci.to_markdown())
ci

In [ ]:
pc = pd.concat({"LightGBM (argmax)": S.per_class_table(fit, "LightGBM|clf"),
                "LightGBM (F1-tuned cutoffs)": S.per_class_table(fit, "LightGBM (F1-tuned cutoffs)|clf"),
                "LightGBM (high-recall cutoff)": S.per_class_table(fit, "LightGBM (high-recall cutoff)|clf"),
                "Random forest": S.per_class_table(fit, "Random forest|clf")})
pc.to_csv(C.REPORTS / "severity_per_class_test.csv"); (C.REPORTS / "severity_per_class_test.md").write_text(pc.to_markdown())
pc

### Operating point for high-severity outages
Accuracy alone rewards predicting "low" (72% of test rows). We report how well the model ranks high-severity outages (AUC, average precision) and choose the cutoff on purpose: the smallest P(high) that catches 50% of 2020's high-severity outages, then apply it unchanged to 2021-22.

In [ ]:
op = S.operating_table(fit)
op.to_csv(C.REPORTS / "severity_operating_point.csv"); (C.REPORTS / "severity_operating_point.md").write_text(op.to_markdown())
print("P(high) cutoff fit on 2020:", fit["high_cutoff"]); op

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
j = list(fit["classes"]).index("high")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for (split, label), col in zip([("val", "2020 (validation)"), ("test", "2021-22 (test)")], viz.CATEGORICAL):
    y = fit["truth"][split].severity.astype(str).to_numpy() == "high"
    prec, rec, _ = precision_recall_curve(y, fit["proba"][split][:, j])
    ax.plot(rec, prec, color=col, label=f"{label}, AP = {average_precision_score(y, fit['proba'][split][:, j]):.2f}")
    ax.axhline(y.mean(), color=col, linestyle=":", linewidth=1)
yt = fit["truth"]["test"].severity.astype(str).to_numpy() == "high"
pt = fit["pred"]["test"]["LightGBM (high-recall cutoff)|clf"].astype(str).to_numpy() == "high"
ax.scatter([(pt & yt).sum() / yt.sum()], [(pt & yt).sum() / pt.sum()], color=viz.CATEGORICAL[1], zorder=3, s=40,
           label="chosen cutoff on test")
ax.set_xlabel("Recall (share of high-severity outages caught)"); ax.set_ylabel("Precision")
ax.set_title("LightGBM: precision-recall for high severity (dotted = base rate)"); ax.legend()
viz.save(fig, "06b_high_severity_pr_curve"); fig

In [ ]:
hp = S.hyperparameter_table()
hp.to_csv(C.REPORTS / "hyperparameters_severity.csv", index=False)
(C.REPORTS / "hyperparameters_severity.md").write_text(hp.to_markdown(index=False))
hp

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
te = fit["truth"]["test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ConfusionMatrixDisplay.from_predictions(te.severity.astype(str), fit["pred"]["test"]["LightGBM|clf"],
    labels=["low", "medium", "high"], normalize="true", cmap=viz.SEQUENTIAL_CMAP, ax=axes[0],
    values_format=".2f", colorbar=False)
axes[0].set_title("LightGBM severity class (row-normalized, test)"); axes[0].grid(False)
p = fit["pred"]["test"]["LightGBM|reg"]
axes[1].scatter(te.log_peak_pct, p, s=6, alpha=0.25, color=viz.CATEGORICAL[0], linewidths=0)
lim = [0, max(te.log_peak_pct.max(), p.max())]
axes[1].plot(lim, lim, color=viz.MUTED, linewidth=1, linestyle="--")
axes[1].set_xlabel("Actual log(1 + % out)"); axes[1].set_ylabel("Predicted")
axes[1].set_title("LightGBM regression: predicted vs actual (test)")
viz.save(fig, "06_severity_test_diagnostics"); fig

In [ ]:
# per storm type: does the best model beat the baseline everywhere?
pt = fit["pred"]["test"]
by_type = pd.DataFrame({
    "n": te.groupby("storm_type").size(),
    "baseline MAE": (te.log_peak_pct - pt["Baseline (storm type)|reg"]).abs().groupby(te.storm_type).mean(),
    "LightGBM MAE": (te.log_peak_pct - pt["LightGBM|reg"]).abs().groupby(te.storm_type).mean(),
}).round(3).sort_values("n", ascending=False)
by_type

## SHAP: what drives predicted severity?

In [ ]:
import shap
sv, X = S.shap_values(fit, n=5000)
plt.figure()
shap.plots.beeswarm(sv, max_display=15, show=False, color=viz.SEQUENTIAL_CMAP)
plt.title("SHAP values, LightGBM severity regressor (test sample)")
fig = plt.gcf(); viz.save(fig, "07_shap_summary"); fig

In [ ]:
imp = pd.Series(np.abs(sv.values).mean(0), index=X.columns).sort_values(ascending=False)
imp.head(15).round(4).to_frame("mean |SHAP|")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, feat in zip(axes, ["max_wind_kts", "county_hist_log_peak"]):
    j = list(X.columns).index(feat)
    ax.scatter(X[feat], sv.values[:, j], s=6, alpha=0.3, color=viz.CATEGORICAL[0], linewidths=0)
    ax.axhline(0, color=viz.AXIS, linewidth=1)
    ax.set_xlabel(feat); ax.set_ylabel("SHAP value"); ax.set_title(f"SHAP dependence: {feat}")
viz.save(fig, "08_shap_dependence"); fig